# 04 Clustering Resolution — сборка групп из пар

Эта тетрадка идёт после `03_matching_comparison.ipynb`.

В предыдущей тетрадке мы решали задачу на уровне пары: связаны два товара или нет. Здесь мы смотрим следующий уровень: если соединить все связанные пары в граф, какие группы товаров получатся.

Проще говоря:

- пара говорит: товар A похож на товар B;
- граф собирает из таких связей цепочки и группы;
- группа показывает возможную товарную семью или конкретную фасовку.


## Две разные группы, которые нельзя путать

`family` — товарная семья или базовый товар. В неё попадают положительные пары модели `exact_duplicate`, включая случаи, где отличается фасовка. Например, один кетчуп 260 г и набор из 6 таких кетчупов могут быть одной семьёй.

`pack` — конкретная фасовка. Сюда должны попадать только пары из той же семьи, у которых совпадает deterministic pack signature: вес единицы и общий pack/multipack.

Почему это важно: модель отвечает только на вопрос «тот же базовый товар или другой товар», а фасовку мы достаём после модели правилами.


## Ограничение этой тетрадки

Сейчас мы строим граф только по размеченным парам из gold-set. Это не полный граф по всей категории `Соусы`.

Значит, цифры здесь — это проверка идеи на сэмпле, а не финальная кластеризация всех товаров.

Полная кластеризация появится только после того, как выбранный метод будет прогнан по всему файлу кандидатов.


## Блок кода 1. Подготовка окружения

Эта ячейка подключает библиотеки, находит корень проекта и импортирует функции для сборки групп.

Главные функции:

- `build_components` — собирает группы из связанных пар;
- `same_pack_signature_mask` — отмечает пары с одинаковой фасовкой по deterministic полям;
- `add_component_flags` — добавляет к каждой паре информацию, оказались ли товары в одной группе;
- `component_size_summary` — показывает размеры получившихся групп.


In [1]:
from __future__ import annotations

from pathlib import Path
import os
import sys

from IPython.display import display
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from research.dedup import (
    FAMILY_EDGE_LABELS,
    PACK_EDGE_LABELS,
    ComponentConfig,
    add_component_flags,
    build_components,
    component_size_summary,
    same_pack_signature_mask,
)

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 160)


## Блок кода 2. Пути и выбор метода

Здесь задаются файлы, которые пришли из тетрадки `03`.

Самое важное:

- `PREDICTIONS_PATH` — все предсказания методов по парам;
- `SUMMARY_PATH` — таблица с качеством методов;
- `METHOD_FROM_ENV` — если нужно вручную выбрать метод для графа;
- `EVAL_SPLIT` — на какой части смотреть качество, обычно `test`.


In [2]:
DATA_DIR = PROJECT_ROOT / "research" / "dedup" / "data"
PREDICTIONS_PATH = DATA_DIR / "matching_predictions_sauces.csv"
SUMMARY_PATH = DATA_DIR / "matching_summary_sauces.csv"
COMPONENTS_PATH = DATA_DIR / "clustering_components_sauces.csv"
PAIR_EVAL_PATH = DATA_DIR / "clustering_pair_eval_sauces.csv"

METHOD_FROM_ENV = os.environ.get("DEDUP_CLUSTERING_METHOD")
EVAL_SPLIT = os.environ.get("DEDUP_CLUSTERING_EVAL_SPLIT", "test")

print(f"Predictions path: {PREDICTIONS_PATH}")
print(f"Summary path: {SUMMARY_PATH}")
print(f"Requested method: {METHOD_FROM_ENV or '<best calibrated test method>'}")
print(f"Eval split: {EVAL_SPLIT}")


Predictions path: /Users/exoldoff/.codex/worktrees/13ef/mpstats/research/dedup/data/matching_predictions_sauces.csv
Summary path: /Users/exoldoff/.codex/worktrees/13ef/mpstats/research/dedup/data/matching_summary_sauces.csv
Requested method: <best calibrated test method>
Eval split: test


## Блок кода 3. Загрузка предсказаний и выбор метода

Эта ячейка читает результаты `03` и выбирает метод, по которому будем строить граф.

Если метод не задан вручную, выбирается лучший текущий метод по `macro_f1` на `test`.

В выводе смотри:

- какой метод выбран;
- сколько пар он предсказал каждым классом;
- как предсказания распределены между `dev` и `test`.


In [ ]:
def _load_predictions() -> tuple[pd.DataFrame, pd.DataFrame]:
    if not PREDICTIONS_PATH.exists():
        raise FileNotFoundError(
            f"{PREDICTIONS_PATH} not found. Run notebooks/03_matching_comparison.ipynb first."
        )
    predictions = pd.read_csv(PREDICTIONS_PATH)
    summary = pd.read_csv(SUMMARY_PATH) if SUMMARY_PATH.exists() else pd.DataFrame()
    return predictions, summary


def _select_method(predictions: pd.DataFrame, summary: pd.DataFrame) -> str:
    if METHOD_FROM_ENV:
        return METHOD_FROM_ENV
    if not summary.empty:
        candidates = summary[(summary["mode"].eq("calibrated")) & (summary["eval_split"].eq("test"))]
        if not candidates.empty:
            return str(candidates.sort_values("macro_f1", ascending=False).iloc[0]["method"])
    return str(predictions["method"].iloc[0])


predictions, matching_summary = _load_predictions()
selected_method = _select_method(predictions, matching_summary)
method_pairs = predictions[(predictions["method"].eq(selected_method)) & (predictions["mode"].eq("calibrated"))].copy()

if method_pairs.empty:
    raise ValueError(f"No calibrated predictions found for method={selected_method!r}")

print(f"Selected method: {selected_method}")
if not matching_summary.empty:
    display(matching_summary)
display(method_pairs["predicted_label"].value_counts().rename_axis("predicted_label").reset_index(name="pairs"))
display(pd.crosstab(method_pairs["eval_split"], method_pairs["predicted_label"]))


## Блок кода 4. Сборка настоящих и предсказанных групп

Эта ячейка строит четыре набора групп:

- настоящие `family` группы по твоей разметке;
- предсказанные `family` группы по меткам метода;
- настоящие `pack` группы по твоей разметке и pack-правилам;
- предсказанные `pack` группы по меткам метода и pack-правилам.

Для `family` связью считается binary-positive label `exact_duplicate`.

Для `pack` связью считается `exact_duplicate` плюс совпавший `same_pack_signature`.

В таблице `component_summary` смотри, сколько групп получилось и сколько из них имеют больше одного товара.


In [4]:
def _node_catalog(pairs: pd.DataFrame) -> pd.DataFrame:
    left_cols = {
        "raw_record_id_a": "node_id",
        "marketplace_a": "marketplace",
        "sku_a": "sku",
        "title_a": "title",
        "brand_a": "brand",
        "unit_amount_a": "unit_amount",
        "total_amount_a": "total_amount",
        "multipack_count_a": "multipack_count",
    }
    right_cols = {key.replace("_a", "_b"): value for key, value in left_cols.items()}
    left = pairs[[column for column in left_cols if column in pairs.columns]].rename(columns=left_cols)
    right = pairs[[column for column in right_cols if column in pairs.columns]].rename(columns=right_cols)
    return pd.concat([left, right], ignore_index=True).drop_duplicates("node_id").reset_index(drop=True)


true_family_config = ComponentConfig(label_col="label", component_col="true_family_id")
pred_family_config = ComponentConfig(label_col="predicted_label", component_col="pred_family_id")
true_pack_config = ComponentConfig(label_col="label", component_col="true_pack_id")
pred_pack_config = ComponentConfig(label_col="predicted_label", component_col="pred_pack_id")

method_pairs["same_pack_signature"] = same_pack_signature_mask(method_pairs)

true_family_components = build_components(method_pairs, edge_labels=FAMILY_EDGE_LABELS, config=true_family_config)
pred_family_components = build_components(method_pairs, edge_labels=FAMILY_EDGE_LABELS, config=pred_family_config)
true_pack_components = build_components(
    method_pairs,
    edge_labels=PACK_EDGE_LABELS,
    config=true_pack_config,
    edge_mask=method_pairs["same_pack_signature"],
)
pred_pack_components = build_components(
    method_pairs,
    edge_labels=PACK_EDGE_LABELS,
    config=pred_pack_config,
    edge_mask=method_pairs["same_pack_signature"],
)

component_summary = pd.DataFrame([
    {"graph": "true_family", "components": true_family_components["true_family_id"].nunique(), "multi_node_components": int((component_size_summary(true_family_components, component_col="true_family_id")["nodes"] > 1).sum())},
    {"graph": "pred_family", "components": pred_family_components["pred_family_id"].nunique(), "multi_node_components": int((component_size_summary(pred_family_components, component_col="pred_family_id")["nodes"] > 1).sum())},
    {"graph": "true_pack", "components": true_pack_components["true_pack_id"].nunique(), "multi_node_components": int((component_size_summary(true_pack_components, component_col="true_pack_id")["nodes"] > 1).sum())},
    {"graph": "pred_pack", "components": pred_pack_components["pred_pack_id"].nunique(), "multi_node_components": int((component_size_summary(pred_pack_components, component_col="pred_pack_id")["nodes"] > 1).sum())},
])

display(component_summary)
display(component_size_summary(pred_family_components, component_col="pred_family_id").head(15))
display(component_size_summary(pred_pack_components, component_col="pred_pack_id").head(15))


,graph,components,multi_node_components
0,true_family,568,137
1,pred_family,560,144
2,true_pack,644,71
3,pred_pack,668,48


,pred_family_id,nodes
0,44,4
1,76,4
2,42,3
3,83,3
4,91,3
5,156,3
6,159,3
7,285,3
8,286,3
9,320,3


,pred_pack_id,nodes
0,8,2
1,10,2
2,14,2
3,15,2
4,18,2
5,29,2
6,37,2
7,39,2
8,40,2
9,58,2


## Блок кода 5. Проверка графа на уровне размеченных пар

Эта ячейка отвечает на вопрос: если смотреть только пары из gold-set, правильно ли граф связал товары.

Метрики здесь похожи на обычную проверку классификации, но уже для связей в графе:

- `precision` — из всех связей, которые граф сделал, сколько были правильными;
- `recall` — из всех настоящих связей, сколько граф нашёл;
- `false_links` — лишние связи, самые опасные для группировки;
- `missed_links` — пропущенные связи.

Отдельно считаются `family` и `pack`, потому что это разные уровни строгости.


In [5]:
pair_eval = add_component_flags(
    method_pairs,
    true_family_components,
    config=true_family_config,
    same_component_col="true_same_family",
)
pair_eval = add_component_flags(
    pair_eval,
    pred_family_components,
    config=pred_family_config,
    same_component_col="pred_same_family",
)
pair_eval = add_component_flags(
    pair_eval,
    true_pack_components,
    config=true_pack_config,
    same_component_col="true_same_pack",
)
pair_eval = add_component_flags(
    pair_eval,
    pred_pack_components,
    config=pred_pack_config,
    same_component_col="pred_same_pack",
)


def _binary_link_report(frame: pd.DataFrame, *, true_col: str, pred_col: str, scope: str) -> dict[str, object]:
    true_link = frame[true_col].astype(bool)
    pred_link = frame[pred_col].astype(bool)
    tp = int((true_link & pred_link).sum())
    fp = int((~true_link & pred_link).sum())
    fn = int((true_link & ~pred_link).sum())
    tn = int((~true_link & ~pred_link).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "scope": scope,
        "eval_split": frame["eval_split"].iloc[0] if frame["eval_split"].nunique() == 1 else "all",
        "pairs": len(frame),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "true_positive_links": tp,
        "false_links": fp,
        "missed_links": fn,
        "true_negative_links": tn,
    }


eval_pairs = pair_eval[pair_eval["eval_split"].eq(EVAL_SPLIT)].copy()
if eval_pairs.empty:
    eval_pairs = pair_eval.copy()

link_report = pd.DataFrame([
    _binary_link_report(eval_pairs, true_col="true_same_family", pred_col="pred_same_family", scope="family"),
    _binary_link_report(eval_pairs, true_col="true_same_pack", pred_col="pred_same_pack", scope="pack"),
])
display(link_report)


,scope,eval_split,pairs,precision,recall,f1,true_positive_links,false_links,missed_links,true_negative_links
0,family,test,151,0.594203,0.694915,0.640625,41,28,18,64
1,pack,test,151,0.578947,0.379310,0.458333,11,8,18,114


## Блок кода 6. Примеры ошибок графа

Эта ячейка показывает конкретные пары, где граф ошибся.

`false links` — товары не должны быть в одной группе, но граф их связал. Это опаснее всего.

`missed links` — товары должны быть связаны, но граф их разделил. Это неприятно, но обычно менее опасно, чем ложная склейка разных товаров.

Эти примеры полезны для понимания, какие случаи должен лучше обрабатывать следующий метод.


In [ ]:
def _show_link_examples(frame: pd.DataFrame, *, true_col: str, pred_col: str, title: str) -> None:
    false_links = frame[(~frame[true_col].astype(bool)) & frame[pred_col].astype(bool)].copy()
    missed_links = frame[frame[true_col].astype(bool) & (~frame[pred_col].astype(bool))].copy()
    columns = [
        "label",
        "predicted_label",
        "score",
        "title_a",
        "title_b",
        "brand_a",
        "brand_b",
        "unit_amount_a",
        "unit_amount_b",
        "multipack_count_a",
        "multipack_count_b",
        "same_pack_signature",
    ]
    print(f"{title}: false links={len(false_links)}, missed links={len(missed_links)}")
    if not false_links.empty:
        display(false_links[[column for column in columns if column in false_links.columns]].head(12))
    if not missed_links.empty:
        display(missed_links[[column for column in columns if column in missed_links.columns]].head(12))


_show_link_examples(eval_pairs, true_col="true_same_family", pred_col="pred_same_family", title="Family graph")
_show_link_examples(eval_pairs, true_col="true_same_pack", pred_col="pred_same_pack", title="Pack graph")


## Блок кода 7. Сохранение групп и примеры крупных групп

Эта ячейка собирает таблицу товаров с номерами предсказанных и настоящих групп.

В выводе показываются примеры крупных предсказанных семей: по ним удобно глазами проверить, не попали ли в одну группу разные вкусы или разные типы товара.

После этого сохраняются два CSV:

- `clustering_components_sauces.csv` — товары и номера групп;
- `clustering_pair_eval_sauces.csv` — пары с признаками, попали ли они в одну группу.


In [7]:
node_catalog = _node_catalog(method_pairs)
components_export = (
    node_catalog
    .merge(pred_family_components, on="node_id", how="left")
    .merge(pred_pack_components, on="node_id", how="left")
    .merge(true_family_components, on="node_id", how="left")
    .merge(true_pack_components, on="node_id", how="left")
)

large_pred_families = component_size_summary(pred_family_components, component_col="pred_family_id")
large_pred_families = large_pred_families[large_pred_families["nodes"] > 1].head(10)
family_examples = components_export[components_export["pred_family_id"].isin(large_pred_families["pred_family_id"])].copy()
family_examples = family_examples.sort_values(["pred_family_id", "title"])

display(family_examples[["pred_family_id", "pred_pack_id", "marketplace", "sku", "brand", "title", "unit_amount", "total_amount", "multipack_count"]].head(40))

components_export.to_csv(COMPONENTS_PATH, index=False)
pair_eval.to_csv(PAIR_EVAL_PATH, index=False)
print(f"Saved components: {COMPONENTS_PATH} ({len(components_export)} rows)")
print(f"Saved pair eval: {PAIR_EVAL_PATH} ({len(pair_eval)} rows)")


,pred_family_id,pred_pack_id,marketplace,sku,brand,title,unit_amount,total_amount,multipack_count
377,42,108,Ozon,514069297,азбука продуктов,"Натуральный сок лайма прямого отжима АЗБУКА ПРОДУКТОВ основа для коктейлей и напитков, заправка для салата и соуса, приправа для рыбы и мяса 200мл",NaN,NaN,1.0
15,42,108,Ozon,194523239,азбука продуктов,"Натуральный сок лимона прямого отжима АЗБУКА ПРОДУКТОВ основа для коктейлей и напитков, заправка для салата и соуса, приправа для рыбы и мяса 1л",1.00,1.00,1.0
303,42,44,Ozon,1560707793,азбука продуктов,"Натуральный сок лимона прямого отжима АЗБУКА ПРОДУКТОВ основа для коктейлей и напитков, заправка для салата и соуса, приправа для рыбы и мяса 200мл*3шт",0.20,0.60,3.0
267,44,47,Ozon,1574067581,буздякский,Соус томатный Башкирский Буздякский - 3 шт x 670г,0.67,2.01,3.0
682,44,49,Ozon,1574082524,буздякский,Соус томатный Краснодарский Буздякский - 4 шт x 670г,NaN,NaN,4.0
626,44,48,Ozon,1574082401,буздякский,Соус томатный Краснодарский Буздякский - 6 шт x 670г,NaN,NaN,6.0
315,44,46,Ozon,1574057210,буздякский,Соус томатный Татарский Буздякский - 2 шт x 670г,0.67,1.34,2.0
115,76,306,WB,145093365,чим-чим,Заправка корейская для салата Фунчозы 3/60г,0.06,0.06,1.0
346,76,84,Ozon,1785181504,чим-чим,"Корейская заправка для моркови ""ЧИМ-ЧИМ"" 3 шт по 60 гр",0.06,0.18,3.0
341,76,83,Ozon,1785170876,чим-чим,"Корейская заправка для спаржи ""ЧИМ-ЧИМ"" 3 шт по 60 гр",0.06,0.18,3.0


Saved components: /Users/exoldoff/.codex/worktrees/13ef/mpstats/research/dedup/data/clustering_components_sauces.csv (716 rows)
Saved pair eval: /Users/exoldoff/.codex/worktrees/13ef/mpstats/research/dedup/data/clustering_pair_eval_sauces.csv (379 rows)


## Что делать после этой тетрадки

Эта тетрадка показывает, как ошибки на уровне пары превращаются в ошибки группировки.

Если в `03` много ложных склеек, то в `04` они могут стать ещё опаснее: одна неправильная связь способна соединить в одну группу несколько разных товаров.

Поэтому следующий шаг — не перенос в production, а улучшение метода сравнения пар.
